# AI Support Ticket Classifier

### Classifying banking customer support tickets using TF-IDF and Linear SVM.

#### An end-to-end NLP application that classifies banking customer support tickets into one of 77 categories, analyzes sentiment (positive, negative, or  neutral), and assigns a priority level (high, medium, or low). The trained Machine Learning model (Support Vector Classifier) is exposed through a Flask REST API, and containerized using Docker.

Imports:

In [ ]:
# dataset
from datasets import load_dataset
# TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer
# SVM- Classifier
from sklearn.svm import LinearSVC
# Pipeline
from sklearn.pipeline import Pipeline
# ML Classification model evaluation metrics
from sklearn.metrics import accuracy_score, classification_report
# to save trained model
import joblib
# VADER for sentiment analysis
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

Load dataset:

In [ ]:
dataset=load_dataset("mteb/banking77")

In [61]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 9993
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 3076
    })
})


Convert dataset to pandas DataFrame:

In [96]:
# Convert to pandas
train_df=dataset["train"].to_pandas()
test_df=dataset["test"].to_pandas()

In [97]:
print(train_df.head())

                                                text  label    label_text
0                     I am still waiting on my card?     11  card_arrival
1  What can I do if my card still hasn't arrived ...     11  card_arrival
2  I have been waiting over a week. Is the card s...     11  card_arrival
3  Can I track my card while it is in the process...     11  card_arrival
4  How do I know if I will get my card, or if it ...     11  card_arrival


In [98]:
print(train_df.columns)
print(train_df.shape)
print(test_df.shape)

Index(['text', 'label', 'label_text'], dtype='object')
(9993, 3)
(3076, 3)


The dataset is divided into:

* Training samples: 9,993
* Test samples: 3,076
* Categories in label_text: 77

Target variable: categories in the label_text column

In [64]:
print(train_df["label_text"].nunique())

77


In [65]:
print(train_df["label_text"].value_counts())

label_text
card_payment_fee_charged                            187
direct_debit_payment_not_recognised                 182
balance_not_updated_after_cheque_or_cash_deposit    181
wrong_amount_of_cash_received                       180
cash_withdrawal_charge                              177
                                                   ... 
lost_or_stolen_card                                  82
card_swallowed                                       61
card_acceptance                                      59
virtual_card_not_working                             41
contactless_not_working                              35
Name: count, Length: 77, dtype: int64


This is a multiclass classification problem.

Pipeline for TF-IDF and SVM Classifier

In [ ]:
model=Pipeline([("tfidf", TfidfVectorizer()),
    ("svm", LinearSVC())
])

In [ ]:
model.fit(train_df["text"],train_df["label_text"])

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('svm', LinearSVC())])

Predict

In [ ]:
predictions=model.predict(test_df["text"])

In [70]:
print(predictions[:10])

['get_physical_card' 'card_arrival' 'card_arrival' 'card_arrival'
 'card_arrival' 'card_delivery_estimate' 'card_arrival' 'card_arrival'
 'card_arrival' 'card_arrival']


Comparing with the actual answers:

In [71]:
print(test_df["label_text"].head(10).to_list())

['card_arrival', 'card_arrival', 'card_arrival', 'card_arrival', 'card_arrival', 'card_arrival', 'card_arrival', 'card_arrival', 'card_arrival', 'card_arrival']


Model evaluation

In [ ]:
accuracy=accuracy_score(
    test_df["label_text"],
    predictions
)
print("Accuracy:", accuracy)

Accuracy: 0.8946684005201561


In [103]:
print(classification_report(test_df["label_text"],
        predictions))

                                                  precision    recall  f1-score   support

                           Refund_not_showing_up       0.93      0.93      0.93        40
                                activate_my_card       0.97      0.97      0.97        40
                                       age_limit       0.95      1.00      0.98        40
                         apple_pay_or_google_pay       0.98      1.00      0.99        40
                                     atm_support       0.90      0.97      0.94        39
                                automatic_top_up       0.95      0.97      0.96        40
         balance_not_updated_after_bank_transfer       0.67      0.78      0.72        40
balance_not_updated_after_cheque_or_cash_deposit       0.93      0.95      0.94        40
                         beneficiary_not_allowed       0.97      0.88      0.92        40
                                 cancel_transfer       0.97      0.95      0.96        40
         

Overall, the model achieves an 89% accuracy.

Testing on new inputs:

In [ ]:
message="I haven't received my card yet"
prediction=model.predict([message])
print(prediction[0])

card_arrival


In [ ]:
message="My transfer is pending"
prediction=model.predict([message])
print(prediction[0])

pending_transfer


In [104]:
message="I do not see a new transaction"
prediction=model.predict([message])
print(prediction[0])

transfer_not_received_by_recipient


Saving the trained model

In [ ]:
joblib.dump(model,"../models/ticket_classifier.pkl")

['../models/ticket_classifier.pkl']

VADER- for sentiment analysis

In [ ]:
analyzer=SentimentIntensityAnalyzer()

Test:

In [ ]:
text="My card was stolen and I am worried"
scores=analyzer.polarity_scores(text)
print(scores)

{'neg': 0.474, 'neu': 0.526, 'pos': 0.0, 'compound': -0.6597}


'compound': -0.6597 => negative sentiment

In [105]:
text="I am happy with your service"
scores=analyzer.polarity_scores(text)
print(scores)

{'neg': 0.0, 'neu': 0.575, 'pos': 0.425, 'compound': 0.5719}


'compound': 0.5719 => positive sentiment

Sentiment is classified using VADER based on the compound score: 

scores >= 0.05 are classified as positive 

scores <= -0.05 are classified as negative

scores between -0.05 and 0.05 are classified as neutral

In [106]:
def get_sentiment(text):
    score=analyzer.polarity_scores(text)["compound"]
    if score>=0.05:
        return "positive"
    elif score<=-0.05:
        return "negative"
    else:
        return "neutral"

In [ ]:
# positive
print(get_sentiment("I am very happy with the service"))
# negative
print(get_sentiment("This is terrible and frustrating"))
# neutral
print(get_sentiment("When will my card arrive?"))

positive
negative
neutral


Priority of the sentence: 

In [ ]:
def get_priority(text, sentiment):
    urgent_words=["urgent","immediately","stolen",
                  "fraud","blocked","emergency"]
    text=text.lower()
    if sentiment=="negative" and any(word in text for word in urgent_words):
        return "high"
    if sentiment=="negative":
        return "medium"
    return "low"

Test:

In [109]:
text="My card was stolen and I need help immediately"
sentiment=get_sentiment(text)
priority=get_priority(text, sentiment)
print("Sentiment:",sentiment)
print("Priority:",priority)

Sentiment: negative
Priority: high


In [110]:
text="I am disappointed with your service"
sentiment=get_sentiment(text)
priority=get_priority(text, sentiment)
print("Sentiment:",sentiment)
print("Priority:",priority)

Sentiment: negative
Priority: medium


In [111]:
text="When will my card arrive?"
sentiment=get_sentiment(text)
priority=get_priority(text, sentiment)
print("Sentiment:",sentiment)
print("Priority:",priority)

Sentiment: neutral
Priority: low


In [112]:
text="My money is stolen"
sentiment=get_sentiment(text)
priority=get_priority(text, sentiment)
print("Sentiment:",sentiment)
print("Priority:",priority)

Sentiment: negative
Priority: high


Combining all (category classification, sentiment classification, priority classification):

In [ ]:
def analyze_ticket(text):
    category=model.predict([text])[0]
    sentiment=get_sentiment(text)
    priority=get_priority(text, sentiment)
    dict={"category": category,"sentiment": sentiment,
        "priority": priority}
    return dict

Test:

In [113]:
result=analyze_ticket("My card was stolen and I am worried")
print(result)

{'category': 'lost_or_stolen_card', 'sentiment': 'negative', 'priority': 'high'}


In [114]:
result=analyze_ticket("When will my card arrive?")
print(result)

{'category': 'card_delivery_estimate', 'sentiment': 'neutral', 'priority': 'low'}


In [115]:
result=analyze_ticket("I haven't received my card yet")
print(result)

{'category': 'card_arrival', 'sentiment': 'neutral', 'priority': 'low'}
